In [1]:
from transformers import AutoModelForSeq2SeqLM , AutoTokenizer

/Users/zainabfirdaus/git/airflow/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import evaluate

In [3]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

In [4]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

In [8]:
prompt = "Translate to French : How are you? "

In [9]:
tokenized  = tokenizer( prompt,return_tensors="pt")

In [10]:
outputs = model.generate(**tokenized , max_new_tokens=256)

In [11]:
tokenizer.decode( outputs[0] , skip_special_tokens=True)

'Comment est ?'

In [12]:
prediction=tokenizer.decode( outputs[0] , skip_special_tokens=True)

In [7]:
bleu = evaluate.load("bleu")

In [14]:
reference="Comment vas-tu ?"

In [16]:
score = bleu.compute(predictions=[prediction] , references=[reference] )

In [17]:
score

{'bleu': 0.0,
 'precisions': [0.6666666666666666, 0.0, 0.0, 0.0],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0,
 'translation_length': 3,
 'reference_length': 3}

In [18]:
bleu.compute(predictions=[prediction] , references=["Comment est ?"] )

{'bleu': 0.0,
 'precisions': [1.0, 1.0, 1.0, 0.0],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0,
 'translation_length': 3,
 'reference_length': 3}

In [5]:
translation_dataset = [
    {
        "english": "Hello, how are you?",
        "french_references": ["Bonjour, comment ça va ?", "Bonjour, comment allez-vous ?"]
    },
    {
        "english": "Where is the nearest train station?",
        "french_references": ["Où se trouve la gare la plus proche ?", "Où est la gare la plus proche ?"]
    },
    {
        "english": "I would like to order a coffee, please.",
        "french_references": ["Je voudrais commander un café, s'il vous plaît.", "J'aimerais commander un café, s'il vous plaît."]
    },
    {
        "english": "The weather is very beautiful today.",
        "french_references": ["Il fait très beau aujourd'hui.", "Le temps est très beau aujourd'hui."]
    },
    {
        "english": "Can you help me, please?",
        "french_references": ["Pouvez-vous m'aider, s'il vous plaît ?", "Peux-tu m'aider, s'il te plaît ?"]
    },
    {
        "english": "How much does this cost?",
        "french_references": ["Combien ça coûte ?", "Quel est le prix de ceci ?"]
    },
    {
        "english": "I love learning new languages.",
        "french_references": ["J'adore apprendre de nouvelles langues.", "J'aime apprendre de nouvelles langues."]
    },
    {
        "english": "What time does the museum close?",
        "french_references": ["À quelle heure ferme le musée ?", "Le musée ferme à quelle heure ?"]
    },
    {
        "english": "Thank you very much for your kind help.",
        "french_references": ["Merci beaucoup pour votre aide précieuse.", "Merci beaucoup pour votre gentille aide."]
    },
    {
        "english": "Have a safe trip back home.",
        "french_references": ["Bon retour chez vous.", "Faites un bon voyage de retour."]
    }
]


In [7]:
predictions = []
references = []
prompt_template = "Translate to French :{}"
for e in translation_dataset : 
    prompt = prompt_template.format(e['english'])
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(**inputs, max_new_tokens= 1024 )
    prediction = tokenizer.decode( output[0] , skip_special_tokens=True )
    predictions.append(prediction)
    references.append(e['french_references'])


In [ ]:
score= bleu.compute(predictions=predictions , references=references)

In [22]:
score

{'bleu': 0.0,
 'precisions': [0.3709677419354839,
  0.1509433962264151,
  0.022727272727272728,
  0.0],
 'brevity_penalty': 0.9682566771439106,
 'length_ratio': 0.96875,
 'translation_length': 62,
 'reference_length': 64}

## Overall Score

* bleu (0.0): The final translation quality score is zero because there are absolutely no matching 4-gram phrase sequences between the generated French text and the reference translation. [1, 2, 3] 

## Attribute Breakdown

* precisions: This shows that 37% of single words matched, 15% of two-word pairs matched, 2% of three-word phrases matched, and 0% of four-word phrases matched the reference.
* brevity_penalty (0.968): A penalty was applied to lower the overall score because the generated translation is slightly shorter than the ideal reference length. [4, 5] 
* length_ratio (0.969): The generated translation achieves roughly 97% of the total length required by the reference text.
* translation_length (62): The generated French translation contains a total count of 62 words or tokens.
* reference_length (64): The human-provided ideal French translation contains a total count of 64 words or tokens.
 

In [6]:
bert = evaluate.load("bertscore")

> Downloads model models--bert-base-multilingual-cased to 
```
 ~/.cache/huggingface/hub
```

In [8]:
bert_score=bert.compute(predictions=predictions , references=references , lang='fr' )

In [9]:
bert_score

{'precision': [0.0,
  0.7758365273475647,
  0.6913265585899353,
  0.8921284675598145,
  0.7373185157775879,
  0.7034268975257874,
  0.8555250763893127,
  0.7946548461914062,
  0.8046997785568237,
  0.7209588289260864],
 'recall': [0.0,
  0.8021966814994812,
  0.7546464204788208,
  0.9095802903175354,
  0.705806314945221,
  0.7359560132026672,
  0.8931488990783691,
  0.794510006904602,
  0.8310173749923706,
  0.7604542970657349],
 'f1': [0.0,
  0.7887964248657227,
  0.7216001152992249,
  0.9007698893547058,
  0.7212183475494385,
  0.7193238735198975,
  0.8719812631607056,
  0.7945824265480042,
  0.8176468014717102,
  0.7401800751686096],
 'hashcode': 'bert-base-multilingual-cased_L9_no-idf_version=0.3.12(hug_trans=4.57.6)'}

summary for each attribute based on the exact numbers  

* precision: Your model generated highly relevant text that averaged 77.5% accuracy across sentences 2–10, though sentence 1 completely failed at 0.0%.
* recall: Your model successfully captured an average of 79.9% of the reference text's core meaning, peaking at an excellent 90.9% on sentence 4.
* f1: Your overall semantic quality scores are strong, averaging 79.4% (excluding the first sentence's 0.0%), with sentence 4 hitting a near-perfect 90.0%. 
* hashcode: Your evaluation relied specifically on the 9th layer of bert-base-multilingual-cased running on Hugging Face transformers version 4.57.6.
 
 
